In [1]:
import sys
import random
from collections import Counter
from pathlib import Path

sys.path.append("../src")

from mortgage_agent.schemas import Applicant
from mortgage_agent.generator import (
    make_loan,
    make_debts,
    make_income,
    make_clean_applicant,
    TRAPS,
    make_applicant,
    generate_dataset,
    save_dataset,
)

In [2]:
rng = random.Random(42)
for _ in range(3):
    print(make_loan(rng))

print()
rng = random.Random(42)
print(make_loan(rng))

amount=212000.0 property_value=265000.0 monthly_housing_payment=1705.0 estimated_closing_costs=6029.0 property_type='2-4 unit' occupancy='principal residence' underwriting='DU'
amount=504000.0 property_value=720000.0 monthly_housing_payment=4005.0 estimated_closing_costs=19272.0 property_type='1-unit' occupancy='principal residence' underwriting='DU'
amount=244000.0 property_value=305000.0 monthly_housing_payment=1947.0 estimated_closing_costs=7387.0 property_type='1-unit' occupancy='second home' underwriting='DU'

amount=212000.0 property_value=265000.0 monthly_housing_payment=1705.0 estimated_closing_costs=6029.0 property_type='2-4 unit' occupancy='principal residence' underwriting='DU'


In [3]:
rng = random.Random(5)
loan = make_loan(rng)
debts = make_debts(rng)
income = make_income(rng, loan.monthly_housing_payment, debts)

print(loan.monthly_housing_payment)
for d in debts:
    print(d)
print(income)

total = loan.monthly_housing_payment + sum(d.monthly_payment for d in debts)
print(f"DTI: {total / income.verified_monthly_income * 100:.1f}%")

4233.0
kind='credit card' monthly_payment=105.0 balance=2100.0 months_remaining=None
kind='student loan' monthly_payment=300.0 balance=15900.0 months_remaining=53
employer='Northwind Logistics' monthly_income=18200.0 verified_monthly_income=18200.0 years_employed=3.5 pay_frequency='monthly' self_employed=False
DTI: 25.5%


In [4]:
from collections import Counter

rng = random.Random(1)
a = make_clean_applicant(rng, "C001")
print(a.name, "|", a.loan.occupancy, "|", a.credit.score, "|", a.assets.bank_balance, "|", a.expected.decision)

rng = random.Random(2026)
results = Counter(make_clean_applicant(rng, f"T{i}").expected.decision for i in range(1000))
print(results)

Mei Smith | principal residence | 769 | 327000.0 | approve
Counter({'approve': 1000})


In [5]:
rng = random.Random(2026)
for trap in TRAPS:
    results = Counter(make_applicant(rng, "T", trap).expected.decision for _ in range(500))
    print(f"{trap:20} {dict(results)}")

print()
rng = random.Random(3)
a = make_applicant(rng, "A001", "recent_bankruptcy")
print(a.closing_date, a.credit.bankruptcy_chapter, a.credit.bankruptcy_discharge_date)
print(a.expected)

none                 {'approve': 500}
low_score            {'decline': 500}
high_dti             {'decline': 500}
short_debt           {'approve': 500}
recent_bankruptcy    {'decline': 500}
old_bankruptcy       {'approve': 500}
recent_foreclosure   {'decline': 500}
old_foreclosure      {'approve': 500}
deposit_needed       {'approve with conditions': 500}
deposit_not_needed   {'approve': 500}
short_funds          {'decline': 500}
gift_ok              {'approve': 500}
gift_no_letter       {'approve with conditions': 500}
gift_bad_donor       {'decline': 500}
gift_rental          {'decline': 500}
min_contribution_short {'decline': 500}
min_contribution_met {'approve': 500}

2026-12-06 13 2024-12-14
decision='decline' reasons=['Chapter 13 bankruptcy discharged 2024-12-14; the 2-year wait ends 2026-12-14, after the 2026-12-06 closing.'] rule_ids=['B3-5.3-07']


In [6]:
rng = random.Random(2026)
for trap in ["deposit_needed", "deposit_not_needed", "short_funds"]:
    results = Counter(make_applicant(rng, "T", trap).expected.decision for _ in range(500))
    print(f"{trap:20} {dict(results)}")

print()
rng = random.Random(4)
a = make_applicant(rng, "A001", "deposit_needed")
print(a.income.verified_monthly_income, a.assets.bank_balance, a.assets.large_deposits[0].amount)
print(a.expected)

deposit_needed       {'approve with conditions': 500}
deposit_not_needed   {'approve': 500}
short_funds          {'decline': 500}

7000.0 90448 22000.0
decision='approve with conditions' reasons=['Verified funds of $68,448 are short of the $70,548 needed; document the source of $22,000 in large deposits.'] rule_ids=['B3-4.2-02']


In [7]:
rng = random.Random(2026)
gift_traps = ["gift_ok", "gift_no_letter", "gift_bad_donor", "gift_rental", "min_contribution_short", "min_contribution_met"]
for trap in gift_traps:
    results = Counter(make_applicant(rng, "T", trap).expected.decision for _ in range(500))
    print(f"{trap:24} {dict(results)}")

print()
rng = random.Random(8)
a = make_applicant(rng, "A001", "min_contribution_short")
print(a.loan.property_value, a.assets.bank_balance, a.assets.gifts[0].amount)
print(a.expected)

gift_ok                  {'approve': 500}
gift_no_letter           {'approve with conditions': 500}
gift_bad_donor           {'decline': 500}
gift_rental              {'decline': 500}
min_contribution_short   {'decline': 500}
min_contribution_met     {'approve': 500}

490000.0 13400 54300.0
decision='decline' reasons=["Gifts are used on a second home at 90% LTV, so 5% ($24,500) must come from the borrower's own funds; they have $13,400."] rule_ids=['B3-4.3-04']


In [2]:
dataset = generate_dataset()
print(len(dataset))
print(Counter(a.expected.decision for _, a in dataset))
for trap, a in dataset[:3]:
    print(a.applicant_id, a.name, "|", trap, "|", a.expected.decision)

folder = Path("../data/applicants/generated")
save_dataset(dataset, folder)
print(len(list(folder.glob("G*.json"))))

loaded = Applicant.model_validate_json((folder / "G001.json").read_text())
print(loaded == dataset[0][1])

again = generate_dataset()
print(all(a1 == a2 for (_, a1), (_, a2) in zip(dataset, again)))

120
Counter({'decline': 56, 'approve': 48, 'approve with conditions': 16})
G001 Diego Haddad | old_bankruptcy | approve
G002 Diego Nguyen | recent_bankruptcy | decline
G003 Elena Novak | none | approve
120
True
True


In [3]:
import json
from pathlib import Path

from mortgage_agent.schemas import Applicant
from mortgage_agent.calculators import ltv, dti, funds_to_close
from mortgage_agent.rules import required_reserves_months

folder = Path("../data/applicants/generated")
manifest = json.loads((folder / "manifest.json").read_text())


def card(a: Applicant, trap: str) -> None:
    """Print the facts a human needs to check one applicant's answer."""
    loan, c, s = a.loan, a.credit, a.assets
    months = required_reserves_months(a)
    needed = funds_to_close(a) + months * loan.monthly_housing_payment
    print(f"== {a.applicant_id} | trap: {trap} ==")
    print(f"Property: {loan.property_type} {loan.occupancy} | value ${loan.property_value:,.0f} | LTV {ltv(a):.0f}% | closing {a.closing_date}")
    print(f"Score {c.score} | DTI {dti(a):.1f}% | stated income ${a.income.monthly_income:,.0f} | verified ${a.income.verified_monthly_income:,.0f}")
    for d in a.debts:
        print(f"  Debt: {d.kind} ${d.monthly_payment:,.0f}/mo, {d.months_remaining} payments left")
    if c.bankruptcy_chapter:
        print(f"  Bankruptcy: Chapter {c.bankruptcy_chapter}, discharged {c.bankruptcy_discharge_date}")
    if c.foreclosure_date:
        print(f"  Foreclosure: completed {c.foreclosure_date}")
    print(f"Money needed ${needed:,.0f} (reserves {months} mo) | bank ${s.bank_balance:,.0f} | retirement ${s.retirement_balance:,.0f}")
    for d in s.large_deposits:
        print(f"  Deposit: ${d.amount:,.0f}, documented={d.source_documented}")
    for g in s.gifts:
        print(f"  Gift: ${g.amount:,.0f} from {g.donor_relationship}, letter={g.has_gift_letter}")
    print(f"ANSWER: {a.expected.decision} {a.expected.rule_ids}")
    for r in a.expected.reasons:
        print(f"  - {r}")
    print()


picks = ["high_dti", "short_debt", "old_bankruptcy", "recent_foreclosure", "deposit_needed",
         "deposit_not_needed", "short_funds", "gift_no_letter", "gift_bad_donor", "min_contribution_short"]
seen = set()
for row in manifest:
    if row["trap"] in picks and row["trap"] not in seen:
        seen.add(row["trap"])
        applicant = Applicant.model_validate_json((folder / f"{row['applicant_id']}.json").read_text())
        card(applicant, row["trap"])

== G002 | trap: short_debt ==
Property: 1-unit second home | value $555,000 | LTV 70% | closing 2026-11-29
Score 758 | DTI 33.9% | stated income $9,800 | verified $9,800
  Debt: student loan $210/mo, 24 payments left
  Debt: auto loan $2,070/mo, 2 payments left
Money needed $187,419 (reserves 2 mo) | bank $225,000 | retirement $0
ANSWER: approve []

== G004 | trap: gift_bad_donor ==
Property: 1-unit principal residence | value $350,000 | LTV 70% | closing 2026-11-01
Score 786 | DTI 34.3% | stated income $7,900 | verified $7,900
  Debt: student loan $470/mo, 38 payments left
  Debt: student loan $240/mo, 45 payments left
Money needed $114,742 (reserves 0 mo) | bank $101,842 | retirement $0
  Gift: $25,000 from interested party, letter=True
ANSWER: decline ['B3-4.3-04']
  - Funds of $101,842 are short of the $114,742 needed; $25,000 in gifts can't be used because the donor or property type isn't allowed.

== G006 | trap: gift_no_letter ==
Property: 1-unit principal residence | value $270

In [4]:
import json
from pathlib import Path

rules = {r["rule_id"]: r for r in json.loads(Path("../data/processed/b3_rules.json").read_text())}

checks = {
    "B3-6-05": "10",
    "B3-5.3-07": "four years",
    "B3-6-02": "50%",
    "B3-5.1-01": "620",
}
for rule_id, word in checks.items():
    text = rules[rule_id]["text"]
    i = text.find(word)
    print(f"== {rule_id}: {rules[rule_id]['title']} ==")
    print(text[max(0, i - 300): i + 300] if i >= 0 else f"'{word}' not found")
    print()

== B3-6-05: Monthly Debt Obligations ==
tal
documentation to support a payment of less than 5%, the lender must use 5% of the outstanding balance as the
borrower's recurring monthly debt obligation.
For DU loan casefiles, if a revolving debt is provided on the loan application without a monthly payment amount,
DU will use the greater of $10 or 5% of the outstanding balance as the monthly payment when calculating the
total debt-to-income ratio.
Student Loans
If a monthly student loan payment is provided on the credit report, the lender may use that amount for
qualifying purposes. If the credit report does not reflect the correct mont

== B3-5.3-07: Significant Derogatory Credit Events — Waiting Periods and Re-establishing Credit ==
e or dismissal date of the bankruptcy action.
Bankruptcy (Chapter 13)
A distinction is made between Chapter 13 bankruptcies that were discharged and those that were dismissed.
The waiting period required for Chapter 13 bankruptcy actions is measured as follow